# Module 07 — Backpropagation, Explained Slowly

Backprop sounds scary. It's just the **chain rule** used to answer one
question for every weight: *"if I nudge you a little, how much does the final
error change?"* That number is the weight's **gradient** — its share of the blame.

We'll compute it by hand for a tiny network so the mechanism is fully visible.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## The tiniest network
One input `x`, one weight `w`, one bias `b`, a sigmoid, and a target `t`.
Prediction: `a = sigmoid(w*x + b)`. Loss: `L = (a - t)^2`.

We want `dL/dw` — how the loss changes as we wiggle `w`.

In [ ]:
x, t = 1.5, 1.0          # one training example: input 1.5, target 1.0
w, b = 0.2, 0.0

# forward pass, keeping every intermediate value
z = w * x + b
a = sigmoid(z)
L = (a - t) ** 2
print(f"z={z:.4f}  a={a:.4f}  loss={L:.4f}")

## The chain rule, one link at a time
`dL/dw = dL/da * da/dz * dz/dw`. Each factor is easy:
- `dL/da = 2(a - t)`
- `da/dz = a(1 - a)`  (sigmoid derivative)
- `dz/dw = x`

In [ ]:
dL_da = 2 * (a - t)
da_dz = a * (1 - a)
dz_dw = x
dL_dw = dL_da * da_dz * dz_dw
print(f"dL/da = {dL_da:.4f}")
print(f"da/dz = {da_dz:.4f}")
print(f"dz/dw = {dz_dw:.4f}")
print(f"-> dL/dw = {dL_dw:.4f}   (this is the gradient: w's share of the blame)")

## Verify against a numerical gradient
Backprop should match the "wiggle w and measure" estimate. If it does, our
analytic gradient is correct. This finite-difference check is how people debug
real autograd code.

In [ ]:
eps = 1e-6
def loss_for(w_):
    return (sigmoid(w_ * x + b) - t) ** 2
numerical = (loss_for(w + eps) - loss_for(w - eps)) / (2 * eps)
print(f"backprop gradient  = {dL_dw:.6f}")
print(f"numerical gradient = {numerical:.6f}")
print(f"match: {abs(dL_dw - numerical) < 1e-5}")

## Now take a step downhill
Repeat forward -> backward -> update and watch the loss fall. This is training.

In [ ]:
w, b = 0.2, 0.0
lr = 1.0
for step in range(15):
    z = w * x + b
    a = sigmoid(z)
    L = (a - t) ** 2
    # backward
    dz = 2 * (a - t) * a * (1 - a)
    w -= lr * dz * x
    b -= lr * dz
    if step % 3 == 0:
        print(f"step {step:2d}: pred={a:.4f} loss={L:.5f}")
print(f"\nConverged toward the target {t}. That's backprop + gradient descent.")

## The leap to big networks
For a network with millions of weights, backprop applies this SAME chain-rule
bookkeeping automatically, layer by layer, from the output back to the input.
`tiny_net_from_scratch.py` does exactly this for a 2-layer XOR net; PyTorch's
`loss.backward()` does it for you at scale. Nothing new — just more links in the chain.

**Next:** run `mnist_pytorch.py` (after `pip install torch`) to see the framework version.